# Analytic Source-Motion Jerk Demo (`solidfmm`)

This notebook demonstrates the analytic `jerk_mode="accurate"` path:

- exact near-field jerk,
- far-field convective jerk,
- far-field analytic source-motion jerk (`dM -> dL`).


In [ ]:
import time

import jax
import jax.numpy as jnp
import numpy as np

from jaccpot import FMMPreset, FastMultipoleMethod


In [ ]:
def direct_sum_jerk(positions, masses, velocities, G=1.0, softening=1e-3):
    diff = positions[:, None, :] - positions[None, :, :]
    vdiff = velocities[:, None, :] - velocities[None, :, :]
    dist_sq = jnp.sum(diff * diff, axis=-1) + softening**2
    eps = jnp.finfo(positions.dtype).eps
    inv_r = jnp.where(dist_sq > 0, 1.0 / (jnp.sqrt(dist_sq) + eps), 0.0)
    inv_r3 = inv_r / dist_sq
    inv_r5 = inv_r3 / dist_sq
    eye = jnp.eye(positions.shape[0], dtype=bool)
    inv_r3 = jnp.where(eye, 0.0, inv_r3)
    inv_r5 = jnp.where(eye, 0.0, inv_r5)
    rv = jnp.sum(diff * vdiff, axis=-1)
    term = vdiff * inv_r3[..., None] - 3.0 * rv[..., None] * diff * inv_r5[..., None]
    weighted = masses[None, :, None] * term
    return -G * jnp.sum(weighted, axis=1)


In [ ]:
key = jax.random.PRNGKey(7)
k1, k2, k3 = jax.random.split(key, 3)
n = 256

positions = jax.random.uniform(k1, (n, 3), minval=-1.0, maxval=1.0, dtype=jnp.float32)
masses = jax.random.uniform(k2, (n,), minval=0.5, maxval=1.5, dtype=jnp.float32)
velocities = jax.random.uniform(k3, (n, 3), minval=-0.2, maxval=0.2, dtype=jnp.float32)

solver = FastMultipoleMethod(preset=FMMPreset.ACCURATE, basis="solidfmm")


In [ ]:
acc_fast, jerk_fast = solver.compute_accelerations_and_jerk(
    positions,
    masses,
    velocities,
    leaf_size=16,
    max_order=4,
    theta=0.6,
    jerk_mode="fast_approx",
)

acc_acc, jerk_acc = solver.compute_accelerations_and_jerk(
    positions,
    masses,
    velocities,
    leaf_size=16,
    max_order=4,
    theta=0.6,
    jerk_mode="accurate",
)

rel_fast_vs_acc = float(jnp.linalg.norm(jerk_fast - jerk_acc) / (jnp.linalg.norm(jerk_acc) + 1e-12))
print(f"relative ||jerk_fast - jerk_accurate|| / ||jerk_accurate||: {rel_fast_vs_acc:.3e}")


In [ ]:
# Accuracy spot-check at very small theta against direct sum.
acc_ref, jerk_ref = solver.compute_accelerations_and_jerk(
    positions,
    masses,
    velocities,
    leaf_size=16,
    max_order=4,
    theta=1e-4,
    jerk_mode="accurate",
)
jerk_direct = direct_sum_jerk(positions, masses, velocities, G=1.0, softening=1e-3)
rel_err = float(jnp.linalg.norm(jerk_ref - jerk_direct) / (jnp.linalg.norm(jerk_direct) + 1e-12))
print(f"relative jerk error (accurate vs direct): {rel_err:.3e}")


In [ ]:
def timed_call(mode, runs=3):
    times = []
    for _ in range(runs):
        t0 = time.perf_counter()
        _, jerk = solver.compute_accelerations_and_jerk(
            positions,
            masses,
            velocities,
            leaf_size=16,
            max_order=4,
            theta=0.6,
            jerk_mode=mode,
        )
        jax.block_until_ready(jerk)
        times.append(time.perf_counter() - t0)
    return float(np.mean(times))

t_fast = timed_call("fast_approx")
t_acc = timed_call("accurate")
print(f"fast_approx: {t_fast:.4f} s")
print(f"accurate   : {t_acc:.4f} s")
print(f"accurate/fast: {t_acc / t_fast:.3f}x")
